# 01 — Il campione da spiegare e le categorie di parole (tentativo 13, fase 2)

1. **Campione**: articoli delle testate valutate, bilanciati per livello NewsGuard 2026: 60 per livello (300), al più 3
   per testata, presi a giro tra le testate (seed fisso). Gli stessi articoli si spiegano con i due modelli.
2. **Termini di parte**, fold per fold, dalle sole testate di training: log-odds con prior di Dirichlet informativo
   (Monroe et al., 2008) tra testate Right e Left (al più 50 articoli per testata): solo parole piene (via le stopword
   di spaCy e i residui di indirizzi web) usate da almeno 10 testate diverse; le 200 più tipiche di ogni lato con |z| > 3.
3. **Categorie**: ogni token (spaCy, `en_core_web_sm`) va in una categoria, nell'ordine di precedenza di
   `comune_xai.CATEGORIE`: cornice del sito, numeri e date, nomi propri (entità spaCy), fonti e attribuzione (verbi e
   parole di attribuzione), rivolgersi al lettore (prima e seconda persona), cautela ed enfasi (hedge e booster di
   Hyland, 2005), termini di parte, morale e religione (MFD 2.0), sensazionalismo (il 10% delle parole più attivanti
   del NRC VAD, maiuscole, "!"), emozioni (NRC EmoLex), punteggiatura, parole funzione, argomento (gli altri sostantivi), altro.

Uscite in `Results/3-xai/`: `campione.csv`, `termini_di_parte.csv`, `token_campione.csv.gz` (un token per riga, con
posizione, frase e categoria).

In [ ]:
PROVA = False      # True: solo il fold 1 e 2 articoli per livello

In [ ]:
import numpy as np
import pandas as pd
import spacy

from comune_xai import CATEGORIE, RESULTS, VERSIONI, campione, categoria, leggi_lessici, termini_di_parte

OUT = RESULTS / ("prova" if PROVA else ""); OUT.mkdir(parents=True, exist_ok=True)
art = pd.read_csv(VERSIONI / "dataset.csv"); art["text"] = art.text.fillna("")
testate = pd.read_csv(VERSIONI / "fold.csv").set_index("domain")
if PROVA:
    testate_c = testate[testate.fold_gruppo == 1]
    camp = campione(art[art.domain.isin(testate_c.index)], testate_c, per_livello=2)
else:
    camp = campione(art, testate)
camp = camp.assign(fold=camp.domain.map(testate.fold_gruppo), orientamento=camp.domain.map(testate.orientamento))
camp[["article_id", "domain", "fold", "label", "orientamento"]].to_csv(OUT / "campione.csv", index=False)
print(f"campione: {len(camp)} articoli di {camp.domain.nunique()} testate")
print(pd.crosstab(camp.label, camp.orientamento, margins=True).to_string())

In [ ]:
lessici = leggi_lessici()
print(f"MFD 2.0: {len(lessici['morale'])} parole; NRC EmoLex: {len(lessici['emozioni'])}; NRC VAD, attivazione >= {lessici['soglia_attivazione']:.3f}: {len(lessici['attivazione'])}")
DI_PARTE, righe = {}, []
for f in sorted(camp.fold.unique()):
    insieme, destra, sinistra = termini_di_parte(art, testate, f)
    DI_PARTE[f] = insieme
    righe += [{"fold": f, "lato": "Right", "parola": w} for w in destra] + [{"fold": f, "lato": "Left", "parola": w} for w in sinistra]
    print(f"fold {f}: {len(insieme)} termini di parte; Right: {', '.join(destra[:15])}\n        Left: {', '.join(sinistra[:15])}")
pd.DataFrame(righe).to_csv(OUT / "termini_di_parte.csv", index=False)

In [ ]:
nlp = spacy.load("en_core_web_sm")
righe = []
for (aid, dom, f, lab, ori), doc in zip(camp[["article_id", "domain", "fold", "label", "orientamento"]].itertuples(index=False),
                                       nlp.pipe(camp.text.tolist(), batch_size=16, n_process=8)):
    frasi = {t.i: k for k, s in enumerate(doc.sents) for t in s}
    for t in doc:
        if t.is_space:
            continue
        righe.append((aid, dom, f, lab, ori, t.i, t.idx, t.idx + len(t.text), t.text, t.lemma_.lower(), t.pos_, t.ent_type_, frasi[t.i],
                      categoria(t, lessici, DI_PARTE[f])))
tok = pd.DataFrame(righe, columns=["article_id", "domain", "fold", "label", "orientamento", "i", "inizio", "fine", "testo", "lemma", "pos",
                                   "entita", "frase", "categoria"])
tok.to_csv(OUT / "token_campione.csv.gz", index=False)
quote = tok.categoria.value_counts(normalize=True).reindex(CATEGORIE).fillna(0)
print(f"{len(tok)} token; quota di token per categoria:"); quote.round(3)